# 📥 Etapa 0: Ingesta y Persistencia en el Data Lake (S3)

En este notebook, realizamos el primer paso crítico del pipeline de **MLOps**: la preparación y centralización de los datos. No solo generamos el dataset sintético, sino que lo movemos a **MinIO** para que sea accesible de forma persistente por cualquier nodo de entrenamiento en el clúster.

---

### 📋 1. Generación del Dataset Sintético
Creamos un conjunto de datos basado en interacciones reales de clientes de **Movistar Argentina**. El dataset incluye:
* **Entrada:** Texto del cliente (quejas, consultas, felicitaciones).
* **Salida (Label):** Clasificación numérica (0 al 3) según la intención detectada.

### 🔌 2. Conectividad con MinIO (S3)
Configuramos la integración con el servicio de almacenamiento de objetos interno. 
* **Variables de Entorno:** El código está preparado para leer `S3_ENDPOINT` y credenciales, permitiendo una configuración flexible en los nodos de **Elyra**.
* **Addressing Style:** Usamos `path style` para asegurar la compatibilidad con el despliegue local de MinIO en OpenShift.

### ☁️ 3. Validación y Carga (Data Ingestion)
1. **Verificación de Bucket:** El script detecta si el bucket `training-data` existe; de lo contrario, lo crea automáticamente.
2. **Transferencia:** El archivo `t-moviles_dataset.csv` se sube al almacenamiento de objetos.

---
> **Resultado esperado:** Al finalizar, el dataset estará disponible en la nube interna, garantizando que el siguiente paso del pipeline (Entrenamiento) pueda leer los datos de forma distribuida.

In [ ]:
import pandas as pd
import boto3
import os
from botocore.client import Config

data = {
    "text": [
        "No tengo internet en Lanús hace 2 días, un desastre el servicio @MovistarArg",
        "Me quiero pasar a otra compañía, ¿cómo pido el PIN de portabilidad?",
        "Excelente la atención en el local de Av. Cabildo, muy rápidos.",
        "Aumentaron la factura de nuevo y nadie me avisó, me voy a dar de baja.",
        "El técnico nunca vino a instalar el router, me quedé esperando toda la mañana.",
        "¿Me pueden decir cuánto debo de mi línea móvil?",
        "La fibra óptica vuela, muy conforme con los 1000 megas.",
        "Sigo sin señal de 4G en el subte línea D, arreglen esto por favor.",
        "Quiero saber el precio del iPhone 15 con el plan de 10GB.",
        "Hace una semana pedí la baja y me siguen cobrando, son unos estafadores."
    ] * 50,
    "label": [0, 1, 2, 1, 0, 3, 2, 0, 3, 1] * 50
}

df = pd.DataFrame(data)
df = df.sample(frac=1).reset_index(drop=True)
file_name = "t-moviles_dataset.csv"
df.to_csv(file_name, index=False)
print(f"✅ Archivo {file_name} generado localmente.")


# pasar estos datos como variables de entorno
s3_endpoint = os.getenv("S3_ENDPOINT", "http://minio-service.t-moviles-ai.svc.cluster.local:9000")
s3_access_key = os.getenv("S3_ACCESS_KEY", "minio")
s3_secret_key = os.getenv("S3_SECRET_KEY", "minio123")
bucket_name = "training-data"

s3 = boto3.client(
    's3',
    endpoint_url=s3_endpoint,
    aws_access_key_id=s3_access_key,
    aws_secret_access_key=s3_secret_key,
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
    region_name='none'
)

try:
    try:
        s3.head_bucket(Bucket=bucket_name)
    except:
        s3.create_bucket(Bucket=bucket_name)
        print(f"🆕 Bucket '{bucket_name}' creado.")

    s3.upload_file(file_name, bucket_name, file_name)
    print(f"🚀 {file_name} subido exitosamente al bucket '{bucket_name}'.")

except Exception as e:
    print(f"❌ Error al subir a S3: {e}")
    raise e 